In [ ]:
"""
Extract mean aridity index within a 10 km bounding box for each site.

Source: Zomer et al. (2022) Global Aridity Index v3.1
        https://doi.org/10.6084/m9.figshare.7504448

Using Annual ETO version (ai_v31_yr.tif) which represents the mean annual aridity index.
Raw values are stored as uint16 integers multiplied by 10,000.
Divide by 10,000 to get the actual aridity index.
Higher values = more humid; lower values = more arid.

Requirements:
    pip install rasterio numpy pandas
"""

import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import from_bounds

# --- Config ---
SITES_CSV = "/capstone/aridgw/data/openet_merged_timeseries_1km/merged_openet_data.csv"
ARIDITY_TIF = "/capstone/aridgw/data/Global_Aridity_Index/Global-AI_ET0__annual_v3_1/ai_v31_yr.tif"   # path to your local Global-AI_ET0_annual_v3_1 TIF
OUTPUT_CSV = "/capstone/aridgw/data/sites_aridity.csv"
BBOX_SIZES_KM = [1, 2, 10]
AI_SCALE = 10_000.0  # raw values are AI * 10000


def km_to_deg(km: float, latitude: float) -> tuple[float, float]:
    """Convert km offset to degrees lat/lon at a given latitude."""
    lat_deg = km / 111.0
    lon_deg = km / (111.0 * np.cos(np.radians(latitude)))
    return lat_deg, lon_deg


def extract_mean_aridity(src: rasterio.DatasetReader, lat: float, lon: float, bbox_km: float) -> float:
    """Extract mean aridity index within bbox_km around a point."""
    half = bbox_km / 2
    lat_off, lon_off = km_to_deg(half, lat)

    window = from_bounds(
        left=lon - lon_off,
        bottom=lat - lat_off,
        right=lon + lon_off,
        top=lat + lat_off,
        transform=src.transform
    )
    data = src.read(1, window=window).astype(float)
    data[data == 0] = np.nan  # 0 is nodata in this dataset

    if np.all(np.isnan(data)):
        return np.nan

    return float(np.nanmean(data)) / AI_SCALE


def main():
    df = pd.read_csv(SITES_CSV)
    sites = df[["site_id", "latitude", "longitude"]].drop_duplicates().copy()

    print(f"Extracting aridity for {len(sites)} sites at {BBOX_KM} km bbox...")

    with rasterio.open(ARIDITY_TIF) as src:
        sites["aridity_index_mean"] = sites.apply(
            lambda row: extract_mean_aridity(src, row["latitude"], row["longitude"], BBOX_KM),
            axis=1
        )

    result = df.merge(sites[["site_id", "aridity_index_mean"]], on="site_id", how="left")
    result.to_csv(OUTPUT_CSV, index=False)

    print(f"\nSaved: {OUTPUT_CSV}")
    print(result[["site_id", "latitude", "longitude", "aridity_index_mean"]].drop_duplicates().to_string())


if __name__ == "__main__":
    main()

In [ ]:
"""
Extract mean aridity index for multiple bounding box sizes per site.

Source: Zomer et al. (2022) Global Aridity Index v3.1
        https://doi.org/10.6084/m9.figshare.7504448

Raw values are stored as uint16 integers multiplied by 10,000.
Divide by 10,000 to get the actual aridity index.
Higher values = more humid; lower values = more arid.

Requirements:
    pip install rasterio numpy pandas
"""
# Libraries
import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import from_bounds

# Retrieving CSV and TIF from Cyberduck
SITES_CSV = "/capstone/aridgw/data/openet_merged_timeseries_1km/merged_openet_data.csv"
ARIDITY_TIF = "/capstone/aridgw/data/Global_Aridity_Index/Global-AI_ET0__annual_v3_1/ai_v31_yr.tif"   # path to your local Global-AI_ET0_annual_v3_1 TIF
# CSV being saved in CyberDuck
OUTPUT_CSV = "/capstone/aridgw/data/sites_aridity_index.csv"

# Bounding boxes to extract (in km^2)
BBOX_SIZES_KM = [1, 2, 4, 10]  # side lengths (km) producing areas of 1, 4, 16, 100 km²


AI_SCALE = 10_000.0  # raw values are AI * 10000


def km_to_deg(km: float, latitude: float) -> tuple[float, float]:
    """Convert km offset to degrees lat/lon at a given latitude."""
    lat_deg = km / 111.0
    lon_deg = km / (111.0 * np.cos(np.radians(latitude)))
    return lat_deg, lon_deg


def extract_mean_aridity(src: rasterio.DatasetReader, lat: float, lon: float, bbox_km: float) -> float:
    """Extract mean aridity index within bbox_km around a point."""
    half = bbox_km / 2
    lat_off, lon_off = km_to_deg(half, lat)

    window = from_bounds(
        left=lon - lon_off,
        bottom=lat - lat_off,
        right=lon + lon_off,
        top=lat + lat_off,
        transform=src.transform
    )
    data = src.read(1, window=window).astype(float)
    data[data == 0] = np.nan  # 0 is nodata in this dataset

    if np.all(np.isnan(data)):
        return np.nan

    return float(np.nanmean(data)) / AI_SCALE


def main():
    df = pd.read_csv(SITES_CSV)
    sites = df[["site_id", "latitude", "longitude"]].drop_duplicates().copy()

    print(f"Extracting aridity for {len(sites)} sites at bbox sizes: {BBOX_SIZES_KM} km...")

    with rasterio.open(ARIDITY_TIF) as src:
        for km in BBOX_SIZES_KM:
            col = f"AI_{km}km"
            sites[col] = sites.apply(
                lambda row: extract_mean_aridity(src, row["latitude"], row["longitude"], km),
                axis=1
            )
            print(f"  {km} km done")

    ai_cols = [f"AI_{km}km" for km in BBOX_SIZES_KM]
    result = df.merge(sites[["site_id"] + ai_cols], on="site_id", how="left")
    result.to_csv(OUTPUT_CSV, index=False)

    print(f"\nSaved: {OUTPUT_CSV}")
    print(result[["site_id", "latitude", "longitude"] + ai_cols].drop_duplicates().to_string())


if __name__ == "__main__":
    main()

Extracting aridity for 50 sites at bbox sizes: [1, 4, 16, 100] km...
  1 km done
  4 km done
  16 km done
  100 km done

Saved: /capstone/aridgw/data/sites_aridity_index.csv
                    site_id   latitude   longitude  AI_1km    AI_4km   AI_16km  AI_100km
0      KSGS.371852100505801  37.315020 -100.850500  0.2871  0.288970  0.286517  0.284158
21     KSGS.372043101363101  37.344950 -101.610400  0.2302  0.230040  0.229786  0.240523
42     KSGS.372539100142504  37.429490 -100.243400  0.3153  0.314155  0.313279  0.321521
63     KSGS.373331098033301  37.560960  -98.058000  0.4661  0.465795  0.464707  0.468837
84     KSGS.373607100565301  37.598740 -100.949700  0.2782  0.280025  0.279874  0.280556
105    KSGS.374111099070401  37.684480  -99.119250  0.3797  0.380690  0.380673  0.385416
126    KSGS.374125100344101  37.690240 -100.579500  0.3101  0.310110  0.309833  0.304480
147    KSGS.374747100552101  37.794920 -100.922500  0.2908  0.289180  0.287483  0.284218
168    KSGS.3751451004857